In [1]:
'''
LOOP AGENT: 
Sometimes a task isn't a straight line; it's a loop of refinement. 
A user might ask for a plan, but have constraints that require checking and re-planning. For this, the ADK provides the LoopAgent.

The LoopAgent executes a sequence of sub-agents repeatedly until a condition is met.
This is perfect for workflows involving trial and error, like planning a trip with a tight schedule.

Our New Workflow: The Perfectionist Planner

Planner Agent: Proposes an itinerary (e.g., a museum and a restaurant).
Critic Agent: Checks the plan against a constraint (e.g., "Is the travel time between these two places less than 30 minutes?").
Refiner Agent: If the critic finds a problem, this agent takes the feedback and creates a new, improved plan. 
If the critic is happy, it calls a special exit_loop tool to stop the process.
The LoopAgent manages this cycle, ensuring we don't get stuck in an infinite loop by setting a max_iterations limit.
'''

'\nLOOP AGENT: \nSometimes a task isn\'t a straight line; it\'s a loop of refinement. \nA user might ask for a plan, but have constraints that require checking and re-planning. For this, the ADK provides the LoopAgent.\n\nThe LoopAgent executes a sequence of sub-agents repeatedly until a condition is met.\nThis is perfect for workflows involving trial and error, like planning a trip with a tight schedule.\n\nOur New Workflow: The Perfectionist Planner\n\nPlanner Agent: Proposes an itinerary (e.g., a museum and a restaurant).\nCritic Agent: Checks the plan against a constraint (e.g., "Is the travel time between these two places less than 30 minutes?").\nRefiner Agent: If the critic finds a problem, this agent takes the feedback and creates a new, improved plan. \nIf the critic is happy, it calls a special exit_loop tool to stop the process.\nThe LoopAgent manages this cycle, ensuring we don\'t get stuck in an infinite loop by setting a max_iterations limit.\n'

In [6]:
import os
import sys
import  json
import asyncio
import random
import string
from uuid import uuid4
from typing import List,Any
from IPython.display import HTML, Markdown, display

#----------ADK , Agent and Evaluation components Tools Contextimports here------------------

from google.adk.agents import Agent, SequentialAgent, LoopAgent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search, ToolContext
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content , Part

from dotenv import load_dotenv

print(" All libraries are imported!")

 All libraries are imported!


In [7]:
load_dotenv()

True

In [8]:
#Runner to Help run the agent: This is a HELPER function
async def run_agent_query(agent:Agent,query:str,session : Session,user_id: str,is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n Running query for agent: '{agent.name}' in session: '{session.id}'...")
    runner = Runner(
        agent = agent,
        session_service = session_service,
        app_name = agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(user_id = user_id ,session_id = session.id,new_message = Content(parts=[Part(text = query)],role ="user")):
            if not is_router:
                # Let's see what the agent is thinking! through events 
                print(f"EVENT:{event}")
                if event.is_final_response():
                    final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print("\n" + "-"*50)
        print("✅ Final Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")
    return final_response

In [9]:
# --- Initializing Session Service ---
session_service = InMemorySessionService()
my_user_id = "adk_user_001"

In [10]:
db_agent = Agent(
    name = "db_agent",
    model = "gemini-3.5-flash",
    instruction = "You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}"
)

In [ ]:
#-----------------Iterative Workflow--------------

# A tool to signal that the loop should terminate
COMPLETION_PHRASE = "The plan is feasible and meets all constraints."

def exit_loop(tool_context : ToolContext):
    """ 
    Call this function ONLY when the plan is approved, signaling the loop should end.
    """
    print(f"  [Tool Call] exit_loop triggered by {tool_context.agent_name}")
    tool_context.actions.escalate = True
    return {}


# Agent 1: Proposes an initial plan
planner_agent = Agent(
    name = "planner_agent",
    model = "gemini-3.5-flash",
    tools = [google_search],
    instruction = """You are a trip planner. Based on the user's request, propose a single activity and a single restaurant. 
    Output only the names, like: 'Activity: Exploratorium, Restaurant: La Mar'.
    """,
    output_key = "current_plan"
)

# Agent 2 (in loop): Critiques the plan
critic_agent = Agent(
    name = "critic_agent",
    model = "gemini-3.5-flash",
    tools = [google_search],
    instruction = f"""
    You are a logistics expert. Your job is to critique a travel plan. The user has a strict constraint: total travel time must be short.
    Current Plan: {{current_plan}}
    Use your tools to check the travel time between the two locations.
    IF the travel time is over 45 minutes, provide a critique, like: 'This plan is inefficient. Find a restaurant closer to the activity.'
    ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'
    """,
    output_key = "criticism"
)

# Agent 3 (in loop): Refines the plan or exits
refiner_agent = Agent(
    name="refiner_agent",
    model="gemini-3.5-flash", 
    tools=[google_search, exit_loop],
    instruction=f"""
    You are a trip planner, refining a plan based on criticism.
    Original Request: {{session.query}}
    Critique: {{criticism}}
    IF the critique is '{COMPLETION_PHRASE}', you MUST call the 'exit_loop' tool.
    ELSE, generate a NEW plan that addresses the critique. Output only the new plan names, like: 'Activity: de Young Museum, Restaurant: Nopa'.
    """,
    output_key="current_plan"
)

#----------------The LoopAgent orchestrates the critique-refine cycle----------------------
refinement_loop = LoopAgent(
    name="refinement_loop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=3
)
# -------------The SequentialAgent puts it all together------------------------------
iterative_planner_agent = SequentialAgent(
    name="iterative_planner_agent",
    sub_agents=[planner_agent, refinement_loop],
    description="A workflow that iteratively plans and refines a trip to meet constraints."
)